> ## SOLUTION / ANSWER KEY &mdash; Lab 7.6
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-06-draft-a-reply.ipynb`](../lab-06-draft-a-reply.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 7.6 &mdash; Draft: Generate a Grounded Reply

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 25 min &nbsp;|&nbsp; **Day 4 &middot; Module 7 &mdash; Task Automation with AI Agents**

### What you'll do
- Fill a reply template with the gathered order fields
- Keep it grounded -- use only fields you actually have
- Give the reply a consistent voice (tone & sign-off)

> **How this lab works (experiential flow):** read the **Concept**, run the **Demo** to see it work, then complete **Your Turn** by replacing every `___` placeholder. Run the **grader** cell at the end &mdash; it prints `[PASS]` / `[FAIL]` / `[TODO]` and a final `Score`. Aim for a full score.

> **Framework note:** the graded steps are **offline &amp; deterministic** (pure Python stdlib); the agent-assembly labs reuse the **LangChain-shaped shim** from Module 6. Advanced labs end with an **optional** cell that runs the **real** library. You are building the **email-drafting agent** &mdash; the client's Lab 4.1.

**Reference:** [Module 7 slides &mdash; Draft — generate a work product](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 7 labs](./index.html)

In [ ]:
# Setup -- run me first
import os
WORK = "/tmp/biaa-lab-07-06"
os.makedirs(WORK, exist_ok=True)
print("Working dir:", WORK)

## Concept
**Draft** turns gathered context into a work product a human will review (deck slide 9): a reply, a
summary, a proposal. Three keys: **ground it** (feed the real order &amp; template so it's specific,
not generic), **give it a voice** (tone, format, limits), and **keep the human** (a draft is a
suggestion, not a sent message). The failure mode to avoid: a draft that **invents** a date or a
policy because it wasn't grounded.

In [ ]:
class PromptTemplate:
    """Mirrors LangChain: PromptTemplate.from_template(...).format(input=..., ...)."""
    def __init__(self, template): self.template = template
    @classmethod
    def from_template(cls, template): return cls(template)
    def format(self, **kw):
        s = self.template
        for k, v in kw.items():
            s = s.replace("{" + k + "}", str(v))
        return s

# The email agent's context sources: a small orders DB and a set of reply templates.
ORDERS = {
    "4471": {"id": "4471", "name": "Priya", "status": "shipped",    "eta": "Friday",    "carrier": "BlueDart"},
    "5090": {"id": "5090", "name": "Sam",   "status": "processing", "eta": "next week", "carrier": "-"},
}
TEMPLATES = {
    "delivery_delay": "Hi {name}, your order {id} has {status} and is due {eta}. Thanks for your patience.",
    "refund":         "Hi {name}, we've started the refund for order {id}. It will reflect in 5-7 days.",
}

print("template:", TEMPLATES["delivery_delay"])

## Your Turn
Complete `draft_reply`: fill the grounded template from the order &mdash; never invent a field.

In [ ]:
def draft_reply(order, kind):
    template = PromptTemplate.from_template(TEMPLATES[kind])
    # ground the draft in the REAL order fields -- do not invent anything
    return template.format(name=order["name"], id=order["id"], status=order["status"], eta=order["eta"])

try:
    reply = draft_reply(ORDERS['4471'], 'delivery_delay')
    print(reply)
    print('---')
    print('mentions the id? ', '4471' in reply)
    print('grounded ETA?    ', 'Friday' in reply)
except Exception as e:
    print('Fill the blanks, then re-run.', type(e).__name__)

In [ ]:
# === Auto-grader: run after filling the blanks above ===
_results = []
def _rec(label, status, extra=""):
    _results.append(status); print(f"[{status}] {label}" + (f" -- {extra}" if extra else ""))
def expect(label, got, want):
    if got == "___" or got is None: _rec(label, "TODO")
    elif got == want: _rec(label, "PASS")
    else: _rec(label, "FAIL", f"got {got!r}")
def expect_true(label, fn):
    try: _rec(label, "PASS" if fn() else "FAIL")
    except Exception as e: _rec(label, "TODO", type(e).__name__)

expect_true("the draft greets the customer by name", lambda: "Priya" in draft_reply(ORDERS["4471"], "delivery_delay"))
expect_true("the draft names the order id", lambda: "4471" in draft_reply(ORDERS["4471"], "delivery_delay"))
expect_true("the draft is grounded in the real ETA", lambda: "Friday" in draft_reply(ORDERS["4471"], "delivery_delay"))
expect_true("the draft invents no other date", lambda: "Monday" not in draft_reply(ORDERS["4471"], "delivery_delay"))
expect_true("the draft keeps its voice (a sign-off)", lambda: "Thanks" in draft_reply(ORDERS["4471"], "delivery_delay"))

_p = _results.count("PASS")
print(f"\nScore: {_p}/{len(_results)}")
print("All checks passed -- lab complete!" if _p == len(_results) else "Keep going: fill the blanks marked ___ and re-run.")

---
### Key takeaway
A grounded draft is specific and correct; an ungrounded one invents facts. Draft agents are high-value and low-risk because the human still holds the pen -- which is why the email agent is the canonical first real-world lab.

[&#8592; All Module 7 labs](./index.html) &nbsp;&middot;&nbsp; [Module 7 slides](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>